#Proyecto Final - Análisis de Datos
#Formula 1
### 06 - Preparación del Dashboard
**Integrantes:**
- Esquivel, Kevin
- Simmons, Abigail
- Solis, Luis
- Villarreal, Sergio

**Fecha:** Julio 2026

Este notebook genera todas las tablas que alimentan el dashboard público de F1.
El dashboard está diseñado para un usuario no técnico: un aficionado, directivo
o patrocinador que quiere entender el desempeño de pilotos y escuderías
sin necesidad de conocer SQL ni programación.

##Indicadores Resumen — KPIs generales (tarjetas del dashboard)

In [0]:
# INDICADORES RESUMEN
# Estas 4 cifras aparecerán como tarjetas (counters) en la parte
# superior del dashboard. Son el primer vistazo que recibe el usuario.

kpis = spark.sql("""
  SELECT
    (SELECT COUNT(*) FROM piloto)                                      AS total_pilotos,
    (SELECT COUNT(*) FROM escuderia)                                   AS total_escuderias,
    (SELECT COUNT(*) FROM carrera)                                     AS carreras_historicas,
    (SELECT ROUND(SUM(monto_aporte) / 1000000, 1) FROM financiamiento) AS patrocinio_millones_usd
""")

kpis.write.mode("overwrite").saveAsTable("dash_kpis")
display(kpis)
print("Tabla dash_kpis lista")

total_pilotos,total_escuderias,carreras_historicas,patrocinio_millones_usd
50,21,120,4903.6


Tabla dash_kpis lista


## Gráfico 1: Top pilotos (gráfico de barras horizontales)

In [0]:
# Gráfico de barras horizontales.
#Responde: ¿quiénes son los mejores pilotos?
# El usuario puede ver de un vistazo quiénes son los mejores pilotos y cuántos puntos tienen.

top_pilotos = spark.sql("""
  SELECT
    CONCAT(p.nombre_piloto, ' ', p.apellido_piloto)  AS `Piloto`,
    p.nacionalidad                                    AS `País`,
    SUM(r.puntos_obtenidos)                           AS `Puntos_Totales`,
    COUNT(r.id_carrera)                               AS `Carreras_Disputadas`,
    ROUND(AVG(r.posicion_final), 1)                   AS `Posición_Promedio`
  FROM resultado r
  JOIN piloto p ON p.id_piloto = r.id_piloto
  GROUP BY p.nombre_piloto, p.apellido_piloto, p.nacionalidad
  ORDER BY `Puntos_Totales` DESC
  LIMIT 10
""")

top_pilotos.write.mode("overwrite").saveAsTable("dash_top_pilotos")
display(top_pilotos)
print("Tabla dash_top_pilotos lista")

Piloto,País,Puntos_Totales,Carreras_Disputadas,Posición_Promedio
Daniel Ricciardo,Australia,412,59,9.1
Juan Rodriguez,Mexico,383,50,9.0
Franco Colapinto,Argentina,319,56,10.1
Charles Leclerc,Monaco,314,45,8.8
Max Verstappen,Netherlands,304,48,10.4
Andrea Kim,Canada,304,57,10.7
David Coulthard,United Kingdom,292,44,9.1
Valtteri Bottas,Finland,292,55,10.8
Alain Prost,France,290,45,8.7
Lucas Fontaine,France,289,49,10.1


Tabla dash_top_pilotos lista


##Gráfica 2: Patrocinio por sector (gráfico de pastel)

In [0]:
#Gráfico de pastel
#Responde: ¿qué sectores invierten más en F1?
# Distribución del patrocinio por sector de industria

patrocinio_sector = spark.sql("""
  SELECT
    pa.sector_industria                              AS `Sector`,
    COUNT(DISTINCT pa.id_patrocinador)               AS `Numero_de_Patrocinadores`,
    ROUND(SUM(f.monto_aporte) / 1000000, 2)          AS `inversion_millones_usd`
  FROM financiamiento f
  JOIN patrocinador pa ON pa.id_patrocinador = f.id_patrocinador
  GROUP BY pa.sector_industria
  ORDER BY `inversion_millones_usd` DESC
""")

patrocinio_sector.write.mode("overwrite").saveAsTable("dash_patrocinio_sector")
display(patrocinio_sector)
print("Tabla dash_patrocinio_sector lista")

Sector,Numero_de_Patrocinadores,inversion_millones_usd
Seguros,9,909.71
Banca,8,839.39
Bebidas,8,788.35
Telecomunicaciones,8,744.62
Automotriz,6,591.48
Tecnologia,6,584.69
Petroleo,5,445.4


Tabla dash_patrocinio_sector lista


##Gráfica 3: Clustering de escuderías (gráfico de disperción)

In [0]:
# Gráfico de dispersión (scatter). 
# Responde: ¿qué tipo de equipo es cada escudería?
# Usa la tabla guardada en el notebook 05 con los resultados de K-means.


escuderias_viz = spark.sql("""
  SELECT
    nombre                                    AS `Escudería`,
    ROUND(presupuesto / 1000000, 1)           AS `Presupuesto_millones`,
    puntos_obtenidos                          AS `Puntos_Totales`,
    ROUND(monto_aporte / 1000000, 1)          AS `Patrocinio_millones`,
    perfil                                    AS `Perfil_de_Escudería`
  FROM escuderias_cluster
  ORDER BY `Puntos_Totales` DESC
""")

escuderias_viz.write.mode("overwrite").saveAsTable("dash_escuderias_perfil")
display(escuderias_viz)
print("Tabla dash_escuderias_perfil lista")

Escudería,Presupuesto_millones,Puntos_Totales,Patrocinio_millones,Perfil_de_Escudería
Ferrari,151.0,1884.0,235.6,Alto rendimiento
McLaren,558.3,1749.0,376.0,Alto rendimiento
Williams,334.8,1124.0,312.1,Alto rendimiento
Renault,311.9,838.0,233.9,Alto rendimiento
Alfa Romeo,367.9,826.0,194.0,"Alto presupuesto, bajo rendimiento"
Lotus,429.3,748.0,283.1,Alto rendimiento
Ligier,476.2,578.0,147.3,"Alto presupuesto, bajo rendimiento"
Red Bull,354.8,568.0,329.3,Alto rendimiento
Tyrrell,302.1,522.0,240.3,Bajo presupuesto y rendimiento
Toro Rosso,311.3,486.0,156.3,Bajo presupuesto y rendimiento


Tabla dash_escuderias_perfil lista


##Gráfico 4: Circuitos con más lluvia (gráfico de barras verticales)

In [0]:
# Responde: ¿en que circuito llueve más?
# probabilidad_lluvia está en porcentaje (rango real: 8.24 a 72.56)
# NO se multiplica por 100 porque ya viene en esa escala.


circuitos_lluvia = spark.sql("""
  SELECT
    c.nombre                               AS circuito,
    c.ubicacion                            AS pais,
    CASE c.ubicacion
      WHEN 'Belgica'       THEN 'Europa'
      WHEN 'Reino Unido'   THEN 'Europa'
      WHEN 'Espana'        THEN 'Europa'
      WHEN 'Japon'         THEN 'Asia'
      WHEN 'Singapur'      THEN 'Asia'
      WHEN 'Brasil'        THEN 'America'
      WHEN 'Mexico'        THEN 'America'
      ELSE 'Otra'
    END                                    AS region,
    COUNT(ca.id_carrera)                   AS carreras_realizadas,
    ROUND(AVG(ca.temperatura), 1)          AS temperatura_promedio,
    ROUND(AVG(ca.probabilidad_lluvia), 1)  AS prob_lluvia_pct
  FROM carrera ca
  JOIN circuito c ON c.id_circuito = ca.id_circuito
  GROUP BY c.nombre, c.ubicacion
  ORDER BY prob_lluvia_pct DESC
  LIMIT 8
""")

circuitos_lluvia.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("dash_circuitos_lluvia")
display(circuitos_lluvia)
print("✓ Tabla dash_circuitos_lluvia lista")

circuito,pais,region,carreras_realizadas,temperatura_promedio,prob_lluvia_pct
Marina Bay,Singapur,Asia,10,32.0,65.6
Spa-Francorchamps,Belgica,Europa,10,22.8,54.0
Suzuka,Japon,Asia,20,22.4,44.0
Interlagos,Brasil,America,10,30.7,39.6
Hermanos Rodriguez,Mexico,America,10,27.5,35.8
Jerez,Espana,Europa,10,27.7,35.1
Silverstone,Reino Unido,Europa,10,22.2,25.4
Barcelona-Cataluna,Espana,Europa,10,28.4,24.0


✓ Tabla dash_circuitos_lluvia lista


##Verificación de las graficas

In [0]:
# Confirma que las 6 tablas existen y tienen datos antes de construir el dashboard 

tablas_dashboard = [
    "dash_kpis",
    "dash_top_pilotos",
    "dash_patrocinio_sector",
    "dash_escuderias_perfil",
    "dash_circuitos_lluvia"
]

print("TABLAS LISTAS PARA EL DASHBOARD\n")
for tabla in tablas_dashboard:
    try:
        df = spark.table(tabla)
        filas = df.count()
        cols = len(df.columns)
        print(f"  ✓ {tabla}: {filas} filas | Columnas: {', '.join(df.columns)}")
    except Exception as e:
        print(f"  ✗ {tabla}: ERROR — {e}")


TABLAS LISTAS PARA EL DASHBOARD

  ✓ dash_kpis: 1 filas | Columnas: total_pilotos, total_escuderias, carreras_historicas, patrocinio_millones_usd
  ✓ dash_top_pilotos: 2400 filas | Columnas: piloto, pais, anio, puntos, posicion_final
  ✓ dash_patrocinio_sector: 7 filas | Columnas: Sector, Numero_de_Patrocinadores, inversion_millones_usd
  ✓ dash_escuderias_perfil: 21 filas | Columnas: Escudería, presupuesto_millones, Puntos_Totales, Patrocinio_millones, Perfil_de_Escudería
  ✓ dash_circuitos_lluvia: 8 filas | Columnas: circuito, pais, region, carreras_realizadas, temperatura_promedio, prob_lluvia_pct
